# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maison-ux/flyrank-assignament1-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one content item (`content_id`) in the selected monthly analysis window. The analysis uses an intermediate month (March 2026, if available) to evaluate content performance and prioritize items for editorial review. The final month is kept separate as a sealed test period.

In [9]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token disponível:", bool(HF_TOKEN))

Token disponível: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features (variáveis de entrada)
- `content_type`: tipo do conteúdo.
- `search_volume`: volume de pesquisa.
- `competition`: concorrência.
- `competition_level`: nível de concorrência.
- `cpc`: custo por clique.
- `keyword_char_count` e `keyword_token_count`: características da palavra-chave.
- `char_count` e `word_count`: tamanho do conteúdo.
- `content_created_date`, `content_updated_date` e `last_optimized_date`: datas relacionadas ao conteúdo.

### Label / Proxy (resultado a analisar)
- `clicks` e `impressions`: métricas de desempenho que poderão representar o resultado futuro, desde que sejam calculadas em uma janela posterior às features.

### Context (variáveis de contexto)
- `client_has_gsc` e `client_has_ga4`: disponibilidade de integrações.
- `gsc_data_available` e `ga4_data_available`: disponibilidade de dados.
- `report_date`: data de referência da observação.
- `main_intent`: intenção principal de busca.

### Excluded (variáveis excluídas)
- `client_hash_id`, `content_hash_id`, `keyword_hash_id` e `url_hash_id`: identificadores, não serão utilizados como preditores.
- Métricas futuras não serão utilizadas nas features para evitar vazamento de dados.
- `is_deleted`: registros excluídos serão avaliados e retirados quando apropriado.

In [10]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Confirmar autenticação
api.whoami()
print("Autenticação no Hugging Face concluída!")

# Listar arquivos do dataset
arquivos = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for arquivo in arquivos:
    print(arquivo)

Autenticação no Hugging Face concluída!
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.par

In [11]:
from huggingface_hub import hf_hub_download

readme_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="README.md",
    repo_type="dataset",
    token=HF_TOKEN
)

with open(readme_path, "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_query_90d
  data_files: fac

In [12]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

arquivos = [
    "dim_clients.parquet",
    "dim_content.parquet",
    "fact_content_query_90d.parquet",
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
]

for arquivo in arquivos:
    caminho = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename=arquivo,
        repo_type="dataset",
        token=HF_TOKEN
    )

    print("\n" + "=" * 60)
    print("ARQUIVO:", arquivo)
    print(pq.ParquetFile(caminho).schema)


ARQUIVO: dim_clients.parquet
required group field_id=-1 duckdb_schema {
  optional binary field_id=-1 client_hash_id (String);
  optional boolean field_id=-1 is_active;
  optional boolean field_id=-1 has_gsc_access;
  optional boolean field_id=-1 has_ga4_access;
  optional binary field_id=-1 access_profile (String);
  optional int32 field_id=-1 client_created_date (Date);
  optional int32 field_id=-1 client_updated_date (Date);
  optional int32 field_id=-1 gsc_data_start (Date);
  optional int32 field_id=-1 ga4_data_start (Date);
}


ARQUIVO: dim_content.parquet
required group field_id=-1 duckdb_schema {
  optional binary field_id=-1 client_hash_id (String);
  optional binary field_id=-1 content_hash_id (String);
  optional binary field_id=-1 keyword_hash_id (String);
  optional binary field_id=-1 url_hash_id (String);
  optional int64 field_id=-1 keyword_char_count (Int(bitWidth=64, isSigned=true));
  optional int64 field_id=-1 keyword_token_count (Int(bitWidth=64, isSigned=true));
 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
import duckdb

con = duckdb.connect()

# Localizar os arquivos já baixados no Colab
from pathlib import Path

arquivos = list(Path("/root/.cache/huggingface").rglob("*.parquet"))

print("Arquivos Parquet encontrados:", len(arquivos))

for arquivo in arquivos:
    print(arquivo.name, arquivo)

Arquivos Parquet encontrados: 4
dim_content.parquet /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
fact_content_query_90d.parquet /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_query_90d.parquet
dim_clients.parquet /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet
data_0.parquet /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [14]:
from pathlib import Path
import duckdb

# Localizar o arquivo de março de 2026
arquivos_marco = list(
    Path("/root/.cache/huggingface").rglob(
        "fact_content_daily_performance/month=2026-03/*.parquet"
    )
)

if not arquivos_marco:
    print("Arquivo de março de 2026 não encontrado.")
else:
    arquivo_marco = str(arquivos_marco[0])
    print("Arquivo encontrado:", arquivo_marco)

    # Consultar os dados
    con = duckdb.connect()

    df = con.execute("""
        SELECT *
        FROM read_parquet(?)
        LIMIT 5
    """, [arquivo_marco]).df()

    display(df)

Arquivo encontrado: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [15]:
# Verificações do arquivo de março de 2026

import pandas as pd

dados = pd.read_parquet(arquivo_marco)

print("Total de registros:", len(dados))
print("Total de colunas:", len(dados.columns))

print("\nPeríodo dos dados:")
print("Início:", dados["report_date"].min())
print("Fim:", dados["report_date"].max())

print("\nValores ausentes por coluna:")
print(dados.isna().sum().sort_values(ascending=False).head(15))

print("\nVerificação de duplicidade:")
chaves = ["report_date", "client_hash_id", "content_hash_id"]

print(
    "Registros duplicados:",
    dados.duplicated(subset=chaves).sum()
)

Total de registros: 9841378
Total de colunas: 30

Período dos dados:
Início: 2026-03-01
Fim: 2026-03-31

Valores ausentes por coluna:
gsc_avg_position            6230317
ga4_data_available          3018741
ga4_sessions                3018741
ga4_pageviews               3018741
ga4_users                   3018741
ai_chatgpt                  3018741
sessions_ai                 3018741
sessions_paid               3018741
sessions_social             3018741
sessions_referral           3018741
sessions_direct             3018741
sessions_organic            3018741
ga4_total_engagement_sec    3018741
ga4_engaged_sessions        3018741
ai_claude                   3018741
dtype: int64

Verificação de duplicidade:
Registros duplicados: 0


## 4. Data limits

The dataset has important limitations. Client history is unbalanced, so not all clients have the same observation period. GSC and GA4 metrics are not consistently available for every record, resulting in missing values. The March 2026 analysis covers only one monthly partition and cannot represent the entire historical dataset. Query-level data uses anonymized and salted hashes over a fixed 90-day window, so it does not expose raw search queries or client identities. Therefore, the data supports aggregated performance analysis, but it cannot establish complete client behavior or prove causality.

In [16]:
# Verificar disponibilidade das métricas GSC e GA4

print("Disponibilidade das fontes de dados:")

for coluna in ["gsc_data_available", "ga4_data_available"]:
    if coluna in dados.columns:
        print(f"\n{coluna}:")
        print(dados[coluna].value_counts(dropna=False))

# Verificar quantidade de dias observados por cliente
dias_por_cliente = (
    dados.groupby("client_hash_id")["report_date"]
    .nunique()
)

print("\nHistórico observado por cliente em março:")
print(dias_por_cliente.describe())

Disponibilidade das fontes de dados:

gsc_data_available:
gsc_data_available
False    6230317
True     3611061
Name: count, dtype: int64

ga4_data_available:
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64

Histórico observado por cliente em março:
count    55.000000
mean     29.890909
std       4.524308
min       9.000000
25%      31.000000
50%      31.000000
75%      31.000000
max      31.000000
Name: report_date, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.